# Libraries
Import the core Python libraries used for data loading, analysis, plotting, and modeling.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load in Source data
Read the credit risk dataset from the CSV file into a pandas DataFrame so it can be explored and analyzed.

In [ ]:
df = pd.read_csv('../data/credit_risk_dataset.csv')

In [ ]:
display(df.head())

In [ ]:
display(df.describe().T)

In [ ]:
df.shape

In [ ]:
display(df.info())

# Check if there are duplicates
Identify any duplicate rows in the dataset to determine whether duplicate records should be removed before analysis.

In [ ]:
duplicates = df[df.duplicated()]
print(duplicates)

# Check for missing values
Count missing or null values in each column to understand the completeness of the dataset.

In [ ]:
df.isna().sum()

## Observation
Missing values were found in `person_emp_length` and `loan_int_rate`. These will need to be handled (imputed or dropped) before modeling.

# Target Variable Distribution
Count the number of records per loan status to understand the balance of the dataset.

In [ ]:
print(df['loan_status'].value_counts())
print(df['loan_status'].value_counts(normalize=True).mul(100).round(1))

## Observation
The dataset shows a class imbalance with approximately 78% non-default (0) and 22% default (1) loans. This imbalance will need to be addressed during modeling using techniques such as class weighting or resampling.

In [ ]:
plt.figure(figsize=(18, 14))
sns.heatmap(df.corr(numeric_only=True), cmap='coolwarm');
plt.title('Feature Correlation Matrix', fontsize=16)
plt.show()

## Observation
The heatmap reveals several notable correlations. `loan_percent_income` (loan amount as % of income) shows a strong positive correlation with the target `loan_status`, indicating that higher debt-to-income ratios are associated with more defaults. `loan_int_rate` also shows a positive correlation with default risk. `person_income` has a negative correlation with the target, suggesting higher income reduces default risk. `loan_amnt` and `loan_int_rate` are moderately correlated, which is expected as larger loans often come with higher rates.

In [ ]:
# Plot distributions for key numeric features to inspect their shape and variability.
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(df['person_age'], kde=True, ax=axes[0, 0])
axes[0, 0].set_title('Age Distribution')
axes[0, 0].set_xlabel('Age')
axes[0, 0].set_ylabel('Count')

sns.histplot(df['person_income'], kde=True, ax=axes[0, 1])
axes[0, 1].set_title('Income Distribution')
axes[0, 1].set_xlabel('Income')
axes[0, 1].set_ylabel('Count')

sns.histplot(df['loan_amnt'], kde=True, ax=axes[1, 0])
axes[1, 0].set_title('Loan Amount Distribution')
axes[1, 0].set_xlabel('Loan Amount')
axes[1, 0].set_ylabel('Count')

sns.histplot(df['loan_percent_income'], kde=True, ax=axes[1, 1])
axes[1, 1].set_title('Loan Percent Income Distribution')
axes[1, 1].set_xlabel('Loan Percent of Income')
axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## Observation
`person_age` shows a roughly normal distribution centered around late 20s to early 30s, with a slight right skew. `person_income` is heavily right-skewed with most borrowers earning under $100,000, but a long tail of high earners. `loan_amnt` appears relatively uniform with a slight concentration around certain loan amounts (possibly standardized products). `loan_percent_income` shows a right-skewed distribution, indicating most borrowers have moderate debt-to-income ratios, but some have very high ratios which could signal higher risk.

In [ ]:
# Visualize the relationship between income and loan percent income using a hexbin-style joint plot.
sns.jointplot(x='person_income', y='loan_percent_income', data=df, kind='hex', gridsize=30)
plt.xlabel('Income')
plt.ylabel('Loan Percent of Income')
plt.show()

## Observation
There is a clear negative relationship between income and loan percent income. Lower income borrowers tend to have higher loan-to-income ratios, which makes sense as they may need to borrow more relative to their income. The density is concentrated in the low-to-moderate income range with moderate loan percentages. High loan percentages (above 0.4) are more common among lower income borrowers, indicating potential higher risk in this segment.

In [ ]:
# Compare the default rate by home ownership status.
sns.barplot(x='person_home_ownership', y='loan_status', data=df)
plt.title('Default Rate by Home Ownership')
plt.xlabel('Home Ownership')
plt.ylabel('Default Rate')
plt.show()

## Observation
Renters have the highest default rate, followed by those with a mortgage. Owning a home outright or having other ownership types shows lower default rates. This suggests that home ownership status is a meaningful risk indicator, with renters being higher risk compared to homeowners.

In [ ]:
# Compare the default rate by loan intent.
sns.barplot(x='loan_intent', y='loan_status', data=df)
plt.title('Default Rate by Loan Intent')
plt.xlabel('Loan Intent')
plt.ylabel('Default Rate')
plt.xticks(rotation=45)
plt.show()

## Observation
Debt consolidation and medical loans have the highest default rates, while education and home improvement loans show lower default rates. This suggests that the purpose of the loan is a meaningful predictor of default risk. Loans for education may have lower risk because borrowers expect higher future earnings, while medical debt may correlate with financial stress.

In [ ]:
# Count how many records are present in each loan status.
sns.countplot(x='loan_status', data=df)
plt.title('Class Distribution of Loan Status')
plt.xlabel('Loan Status (0 = No Default, 1 = Default)')
plt.ylabel('Count')
plt.show()

## Observation
The dataset is imbalanced with approximately 78% non-default (0) and 22% default (1) loans. This class imbalance will need to be addressed during modeling using techniques such as class_weight='balanced' or resampling methods to ensure the model doesn't simply default to predicting the majority class.

In [ ]:
# Compare key numeric features across loan status with multiple boxplots.
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True)

sns.boxplot(x='loan_status', y='person_age', data=df, ax=axes[0, 0])
axes[0, 0].set_title('Age by Loan Status')
axes[0, 0].set_xlabel('')
axes[0, 0].set_ylabel('Age')

sns.boxplot(x='loan_status', y='person_income', data=df, ax=axes[0, 1])
axes[0, 1].set_title('Income by Loan Status')
axes[0, 1].set_xlabel('')
axes[0, 1].set_ylabel('Income')

sns.boxplot(x='loan_status', y='loan_amnt', data=df, ax=axes[1, 0])
axes[1, 0].set_title('Loan Amount by Loan Status')
axes[1, 0].set_xlabel('Loan Status (0 = No Default, 1 = Default)')
axes[1, 0].set_ylabel('Loan Amount')

sns.boxplot(x='loan_status', y='loan_percent_income', data=df, ax=axes[1, 1])
axes[1, 1].set_title('Loan Percent Income by Loan Status')
axes[1, 1].set_xlabel('Loan Status (0 = No Default, 1 = Default)')
axes[1, 1].set_ylabel('Loan Percent of Income')

plt.tight_layout()
plt.show()

## Observation
The boxplots reveal clear patterns between features and loan default. Defaulters tend to have slightly lower ages on average. Interestingly, defaulters have higher median loan amounts and significantly higher loan-to-income ratios. The `loan_percent_income` shows the strongest separation between default and non-default groups, confirming it as a key risk indicator. Income appears slightly lower for defaulters, but the difference is less pronounced than for loan-related features.